# Ejercicio 9: Uso de la API de Google Gemini

En este ejercicio vamos a aprender a utilizar la API de Google Gemini para Recuperación de Información.

**Corpus:** Rotten Tomatoes Movies and Critic Reviews
**Embeddings:** Gemini `text-embedding-004`
**Recuperación:** Similitud coseno

## 1. Uso básico

El siguiente código sirve para conectarse con la API de Google Gemini de forma básica

In [11]:
import re
import string
import time

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.metrics.pairwise import cosine_similarity
import gensim.downloader as gensim_api
from gensim.models import KeyedVectors

from google import genai

nltk.download('stopwords', quiet=True)
STOPWORDS_EN = set(stopwords.words('english'))

# Configurar cliente Gemini (solo para RAG)
client = genai.Client(api_key=API_GEMINI)
print('Cliente Gemini configurado.')

# Verificar conexión
try:
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents='¿Qué es la Recuperación de Información en 1 oración?'
    )
    print('Gemini:', response.text)
except Exception as e:
    print(f'generate_content: {e}')

Cliente Gemini configurado.
Gemini: La Recuperación de Información es el proceso de buscar y extraer documentos, datos o información pertinente de una gran colección en respuesta a una consulta o necesidad del usuario.


## 2. Retrieval

### 2.1 Cargo el corpus de Rotten Tomatoes

In [12]:
BASE = '../ExamenPrimero/'

df_movies  = pd.read_csv(BASE + 'rotten_tomatoes_movies.csv')
df_reviews = pd.read_csv(BASE + 'rotten_tomatoes_critic_reviews.csv')

# Join reseñas con título de película
df_corpus = df_reviews.merge(
    df_movies[['rotten_tomatoes_link', 'movie_title']],
    on='rotten_tomatoes_link', how='left'
)
df_corpus = df_corpus[df_corpus['review_content'].notna()].reset_index(drop=True)

# Muestra de 2000 docs — manejable para llamadas a la API
SAMPLE_SIZE = 2000
df_sample = df_corpus.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

# Texto del documento: título + crítico + contenido de la reseña
df_sample['texto'] = (
    df_sample['movie_title'].fillna('') + ' ' +
    df_sample['critic_name'].fillna('') + ' ' +
    df_sample['review_content'].fillna('')
).str.strip()

print(f'Corpus cargado: {len(df_sample):,} documentos (muestra del total: {len(df_corpus):,})')
df_sample[['movie_title', 'critic_name', 'review_content']].head(5)

Corpus cargado: 2,000 documentos (muestra del total: 1,064,211)


,movie_title,critic_name,review_content
0,Snatched,Paul Whitington,"It's predictable stuff for the most part, and ..."
1,Eden,Tina Hassannia,Though the film defines Paul's life by his rel...
2,Cold Fish,Virginie Sélavy,"Just as with Suicide Club, the deliberate weir..."
3,Bad Boys,Roger Ebert,"""Bad Boys"" misses its chance at greatness, but..."
4,Amulet,Alissa Wilkinson,"Creepy, bloody, and flat-out weird in places, ..."


### 2.2 Transformo a embeddings

Se usa el modelo `text-embedding-004` de Gemini. Las llamadas se realizan en lotes de 100 textos para respetar los límites de la API.

In [13]:
model_w2v = gensim_api.load('word2vec-google-news-300')

def preprocess(texto: str) -> list[str]:
    texto = texto.lower()
    texto = texto.translate(str.maketrans('', '', string.punctuation))
    texto = re.sub(r'\s+', ' ', texto).strip()
    return [t for t in texto.split() if t not in STOPWORDS_EN and len(t) > 1]

def embed_doc(tokens: list[str]) -> np.ndarray:
    vecs = [model_w2v[t] for t in tokens if t in model_w2v]
    return np.mean(vecs, axis=0) if vecs else np.zeros(model_w2v.vector_size)

df_sample['tokens'] = df_sample['texto'].apply(preprocess)

print('Generando embeddings Word2Vec...')
doc_embeddings = np.array([embed_doc(t) for t in df_sample['tokens']])
print(f'Matriz de embeddings: {doc_embeddings.shape}  (documentos × dimensiones)')

Generando embeddings Word2Vec...
Matriz de embeddings: (2000, 300)  (documentos × dimensiones)


### 2.3 Creo una query y hago la búsqueda

In [14]:
query = 'science fiction movie with advanced technology and space exploration'

print(f'Query: {query}')

query_tokens = preprocess(query)
query_embedding = embed_doc(query_tokens).reshape(1, -1)
print(f'Dimensión del embedding: {query_embedding.shape[1]}')

Query: science fiction movie with advanced technology and space exploration
Dimensión del embedding: 300


Obtengo los 5 documentos más similares a mi query

In [15]:
K = 5

# Similitud coseno entre query y todos los documentos
scores = cosine_similarity(query_embedding, doc_embeddings).flatten()

# Índices ordenados de mayor a menor similitud
top_idx = scores.argsort()[::-1][:K]

rows = []
for rank, idx in enumerate(top_idx, start=1):
    doc = df_sample.iloc[idx]
    rows.append({
        'Ranking'   : rank,
        'Película'  : doc['movie_title'],
        'Crítico'   : doc['critic_name'],
        'Fragmento' : str(doc['review_content'])[:200],
        'Similitud' : round(float(scores[idx]), 4)
    })

df_top5 = pd.DataFrame(rows)
print(f"Top {K} documentos más similares a: '{query}'\n")
df_top5

Top 5 documentos más similares a: 'science fiction movie with advanced technology and space exploration'



,Ranking,Película,Crítico,Fragmento,Similitud
0,1,Elysium,Robert Roten,This is a big budget science fiction film with...,0.5804
1,2,Spider-Man: Into the Spider-Verse,Clarisse Loughrey,[Makes] a strong case that animation is the na...,0.5587
2,3,Manifesto,Peter Bradshaw,There is a hypnotic fascination to this work b...,0.5584
3,4,Sicario,Hugo Hernández Valdivia,A stylish film where the director uses this re...,0.5424
4,5,High Life,Kelli Weston,"For all the film's epic ambitions, mapping int...",0.5397


In [17]:
# RAG: Word2Vec recupera → Gemini genera respuesta
context = '\n'.join(
    f"  {i+1}. [{row['Película']}] {row['Fragmento']}"
    for i, (_, row) in enumerate(df_top5.iterrows())
)

prompt = (
    f'Contexto (reseñas de películas recuperadas):\n{context}\n\n'
    f'Pregunta: {query}\n'
    'Responde en español basándote solo en el contexto dado.'
)

try:
    rag_response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
    )
    print('Respuesta RAG (Gemini 2.5-flash):\n')
    print(rag_response.text)
except Exception as e:
    print(f'RAG Gemini no disponible: {e}')
    print('\nContexto recuperado:')
    print(context)

Respuesta RAG (Gemini 2.5-flash):

Según el contexto, [Elysium] es una película de ciencia ficción de gran presupuesto con algunos buenos efectos especiales. No se menciona explícitamente la exploración espacial en el contexto proporcionado.
